In [4]:
"""
CIFAR-10 Data Preprocessing Pipeline
======================================

Notebook này thực hiện toàn bộ quy trình chuẩn bị dữ liệu cho experiments với noisy labels.
Chạy notebook này MỘT LẦN để tạo tất cả datasets cần thiết cho các noise ratios khác nhau.

Quy trình:
1. Setup và import các thư viện cần thiết
2. Định nghĩa các noise ratios muốn test
3. For mỗi noise ratio:
   - Tạo dataset và CSV files
   - Verify tính đúng đắn của data
   - Run EDA analysis
4. Tổng kết và báo cáo kết quả
"""

# ============================================================================
# CELL 1: Setup và Import
# ============================================================================

import os
import sys
from pathlib import Path
import logging
from datetime import datetime

# Thiết lập CUBLAS để đảm bảo reproducibility
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# Import các thư viện cơ bản
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import sys
import os
from pathlib import Path  # Sử dụng Path để lấy parent dễ dàng và an toàn hơn

# Lấy đường dẫn thư mục parent của current working directory
parent_dir = Path(os.getcwd()).parent

# Thêm vào sys.path (dùng str() để chuyển thành string)
sys.path.insert(0, str(parent_dir))

# Kiểm tra để debug
print("Thư mục parent:", parent_dir)
print("Sys.path cập nhật:", sys.path)


# Import các modules từ project
try:
    from src.config import TrainConfig
    from src.dataset_utils import prepare_cifar_data
    from src.seed_utils import set_global_seed
    from src.eda_analysis import perform_complete_eda
    print("✓ Successfully imported project modules")
except ImportError as e:
    print(f"❌ Failed to import modules: {e}")
    print("Please ensure you're running from the correct directory with src/ folder accessible")
    raise


# Thêm parent directory vào sys.path để import modules
parent_dir = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

print(f"Working directory: {os.getcwd()}")
print(f"Parent directory: {parent_dir}")
print(f"Python path updated: {str(parent_dir) in sys.path}")

# Thiết lập logging để có thể monitor quá trình
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(f'data_preprocessing_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
    ]
)
logger = logging.getLogger(__name__)

# Thiết lập style cho matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n" + "="*80)
print("CIFAR-10 DATA PREPROCESSING PIPELINE")
print("="*80)
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


# ============================================================================
# CELL 2: Configuration và Định Nghĩa Noise Ratios
# ============================================================================

print("\n" + "="*80)
print("CONFIGURATION")
print("="*80)

# Định nghĩa các noise ratios mà bạn muốn tạo datasets
# Bạn có thể thay đổi list này tùy theo nhu cầu experiment
NOISE_RATIOS = [0.2, 0.4, 0.6, 0.8]  # 20%, 40%, 60%, 80% label noise

print(f"\nNoise ratios to process: {NOISE_RATIOS}")
print(f"Total datasets to create: {len(NOISE_RATIOS)}")

# # Thiết lập các parameters chung cho tất cả datasets
# BASE_DATA_DIR = "./data"  # Thư mục gốc chứa images (shared) và CSVs (per noise ratio)
# SEED = 42  # Random seed để reproducibility
# IMG_SIZE = 224  # Kích thước ảnh sau khi resize (224x224 cho ImageNet pretrained models)
# SPLIT_TRAIN = 0.8  # 80% của 50k trainval samples cho training
# SPLIT_VAL = 0.2  # 20% của 50k trainval samples cho validation

# print(f"\nBase data directory: {BASE_DATA_DIR}")
# print(f"Random seed: {SEED}")
# print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
# print(f"Train/Val split: {SPLIT_TRAIN:.0%} / {SPLIT_VAL:.0%}")

# # Tạo base data directory nếu chưa tồn tại
# os.makedirs(BASE_DATA_DIR, exist_ok=True)
# print(f"\n✓ Base directory ready: {BASE_DATA_DIR}")

# Tạo TrainConfig object để sử dụng trong các functions
base_config = TrainConfig()
# base_config.seed = SEED
# base_config.img_size = IMG_SIZE
# base_config.split_train = SPLIT_TRAIN
# base_config.split_val = SPLIT_VAL
# base_config.data_dir = BASE_DATA_DIR

print("\n✓ Configuration completed")


# ============================================================================
# CELL 3: Helper Functions cho Verification
# ============================================================================

def quick_verify_csv(csv_path: str) -> dict:
    """
    Thực hiện quick verification trên một CSV file và return summary statistics.
    
    Args:
        csv_path: Path to CSV file
        
    Returns:
        dict: Dictionary chứa verification results và statistics
    """
    try:
        df = pd.read_csv(csv_path)
        
        # Các checks cơ bản
        checks = {
            'file_loaded': True,
            'row_count': len(df),
            'has_all_columns': all(col in df.columns for col in 
                                  ['index', 'image_path', 'label_noisy', 'label_orig', 
                                   'class_name', 'split', 'noise_flag']),
            'unique_indices': df['index'].nunique() == len(df),
            'label_range_valid': (df['label_orig'].min() >= 0 and df['label_orig'].max() <= 9 and
                                 df['label_noisy'].min() >= 0 and df['label_noisy'].max() <= 9),
            'noise_flag_valid': df['noise_flag'].isin([0, 1]).all(),
            'clean_samples_consistent': (df[df['noise_flag'] == 0]['label_noisy'] == 
                                        df[df['noise_flag'] == 0]['label_orig']).all(),
            'noisy_samples_different': (df[df['noise_flag'] == 1]['label_noisy'] != 
                                       df[df['noise_flag'] == 1]['label_orig']).all() if (df['noise_flag'] == 1).any() else True,
        }
        
        # Statistics
        stats = {
            'total_samples': len(df),
            'clean_samples': (df['noise_flag'] == 0).sum(),
            'noisy_samples': (df['noise_flag'] == 1).sum(),
            'noise_ratio': (df['noise_flag'] == 1).sum() / len(df) if len(df) > 0 else 0,
            'splits': df['split'].value_counts().to_dict(),
            'class_distribution': df['label_orig'].value_counts().sort_index().to_dict()
        }
        
        return {
            'success': all(checks.values()),
            'checks': checks,
            'stats': stats,
            'error': None
        }
        
    except Exception as e:
        return {
            'success': False,
            'checks': {},
            'stats': {},
            'error': str(e)
        }


def print_verification_summary(results: dict, csv_name: str):
    """
    In ra summary của verification results một cách dễ đọc.
    
    Args:
        results: Dictionary từ quick_verify_csv
        csv_name: Tên của CSV file (để hiển thị)
    """
    print(f"\n  📄 {csv_name}:")
    
    if not results['success']:
        print(f"    ❌ FAILED: {results['error']}")
        for check_name, passed in results['checks'].items():
            if not passed:
                print(f"      ❌ {check_name}")
        return
    
    stats = results['stats']
    print(f"    ✓ Total samples: {stats['total_samples']}")
    print(f"    ✓ Clean: {stats['clean_samples']} ({stats['clean_samples']/stats['total_samples']*100:.1f}%)")
    print(f"    ✓ Noisy: {stats['noisy_samples']} ({stats['noisy_samples']/stats['total_samples']*100:.1f}%)")
    print(f"    ✓ Actual noise ratio: {stats['noise_ratio']:.4f}")
    
    # In split distribution
    if 'splits' in stats:
        split_str = ", ".join([f"{k}: {v}" for k, v in stats['splits'].items()])
        print(f"    ✓ Splits: {split_str}")


print("✓ Helper functions defined")


# ============================================================================
# CELL 4: Main Processing Loop - Tạo Datasets cho Tất Cả Noise Ratios
# ============================================================================

print("\n" + "="*80)
print("STARTING DATA PREPARATION FOR ALL NOISE RATIOS")
print("="*80)

# Dictionary để lưu kết quả của mỗi noise ratio
processing_results = {}

# Dictionary để lưu verification results
verification_results = {}

# Progress bar cho toàn bộ quá trình
overall_progress = tqdm(NOISE_RATIOS, desc="Processing noise ratios", position=0)

for noise_ratio in overall_progress:
    overall_progress.set_description(f"Processing noise_ratio={noise_ratio}")
    
    print("\n" + "="*80)
    print(f"PROCESSING NOISE RATIO: {noise_ratio}")
    print("="*80)
    
    # Tạo config cho noise ratio này
    config = TrainConfig()
    config.noise_ratio = noise_ratio
    
    
    # Set random seed để reproducibility
    set_global_seed(config.seed, deterministic=True)
    
    try:
        # ====================================================================
        # Step 1: Tạo Dataset và CSV Files
        # ====================================================================
        print(f"\n{'='*60}")
        print(f"Step 1: Creating dataset and CSV files (noise={noise_ratio})")
        print(f"{'='*60}")
        
        # Check xem CSV files đã tồn tại chưa
        csv_dir = Path(config.data_dir) / "csvs" / f"noise_{noise_ratio}"
        train_csv_exists = (csv_dir / "train.csv").exists()
        print("csv_dir:", csv_dir)
        print("train_csv_exists:", train_csv_exists)
        
        if train_csv_exists:
            print(f"⚠ CSV files for noise_ratio={noise_ratio} already exist")
            print(f"  Location: {csv_dir}")
            print(f"  Skipping data preparation...")
            
            # Load existing paths
            paths = {
                "train_csv": str(csv_dir / "train.csv"),
                "val_csv": str(csv_dir / "val.csv"),
                "test_csv": str(csv_dir / "test.csv"),
            }
        else:
            print(f"Creating new dataset for noise_ratio={noise_ratio}...")
            
            # Gọi prepare_cifar_data để tạo dataset
            # Function này sẽ:
            # - Check và reuse images nếu đã có (nhờ flag file)
            # - Tạo CSV files mới với noise injection
            # - Verify data integrity trước khi save
            # paths = prepare_cifar_data(config, force_download=False)
            
            print(f"\n✓ Dataset created successfully!")
            print(f"  Train CSV: {paths['train_csv']}")
            print(f"  Val CSV: {paths['val_csv']}")
            print(f"  Test CSV: {paths['test_csv']}")
        
        # ====================================================================
        # Step 2: Verify Dataset đã sinh ra
        # ====================================================================
        print(f"\n{'='*60}")
        print(f"Step 2: Verifying dataset (noise={noise_ratio})")
        print(f"{'='*60}")
        
        # Verify từng CSV file
        csv_files = {
            'train.csv': paths['train_csv'],
            'val.csv': paths['val_csv'],
            'test.csv': paths['test_csv']
        }
        
        verification_results[noise_ratio] = {}
        all_verified = True
        
        for csv_name, csv_path in csv_files.items():
            print(f"\nVerifying {csv_name}...")
            results = quick_verify_csv(csv_path)
            verification_results[noise_ratio][csv_name] = results
            
            if results['success']:
                print(f"  ✓ {csv_name} verification PASSED")
                print_verification_summary(results, csv_name)
            else:
                print(f"  ❌ {csv_name} verification FAILED")
                print(f"  Error: {results['error']}")
                all_verified = False
        
        # Special check: test set phải không có noise
        test_results = verification_results[noise_ratio]['test.csv']
        if test_results['success']:
            if test_results['stats']['noisy_samples'] > 0:
                print(f"\n  ❌ WARNING: Test set has {test_results['stats']['noisy_samples']} noisy samples!")
                print(f"     Test set should always be clean.")
                all_verified = False
            else:
                print(f"\n  ✓ Test set is clean (0 noisy samples)")
        
        if all_verified:
            print(f"\n✓ All CSV files verified successfully for noise_ratio={noise_ratio}")
        else:
            print(f"\n⚠ Some verification checks failed for noise_ratio={noise_ratio}")
            print(f"  Please review the output above for details.")
        
        # ====================================================================
        # Step 3: Run EDA Analysis
        # ====================================================================
        print(f"\n{'='*60}")
        print(f"Step 3: Running EDA analysis (noise={noise_ratio})")
        print(f"{'='*60}")
        
        # Tạo thư mục EDA
        eda_dir = Path(config.data_dir) / "eda_analysis" / f"noise_{noise_ratio}"
        eda_dir.mkdir(parents=True, exist_ok=True)
        
        # Check xem EDA đã chạy chưa
        eda_flag = eda_dir / "eda_complete.flag"
        
        if eda_flag.exists():
            print(f"⚠ EDA for noise_ratio={noise_ratio} already completed")
            print(f"  Location: {eda_dir}")
            print(f"  Skipping EDA...")
        else:
            print(f"Running EDA analysis...")
            print(f"This will create:")
            print(f"  - Class distribution plots")
            print(f"  - Confusion matrices (counts and ratios)")
            print(f"  - Noise analysis report")
            
            # Run EDA
            perform_complete_eda(paths, noise_ratio, str(eda_dir))
            
            # Tạo flag file
            eda_flag.touch()
            
            print(f"\n✓ EDA completed successfully!")
            print(f"  Results saved to: {eda_dir}")
        
        # ====================================================================
        # Lưu processing results
        # ====================================================================
        processing_results[noise_ratio] = {
            'success': True,
            'paths': paths,
            'verification_passed': all_verified,
            'eda_dir': str(eda_dir)
        }
        
        print(f"\n{'='*60}")
        print(f"✓ Completed processing for noise_ratio={noise_ratio}")
        print(f"{'='*60}")
        
    except Exception as e:
        print(f"\n❌ Error processing noise_ratio={noise_ratio}: {e}")
        import traceback
        traceback.print_exc()
        
        processing_results[noise_ratio] = {
            'success': False,
            'error': str(e),
            'verification_passed': False
        }
        continue

overall_progress.close()

print("\n" + "="*80)
print("DATA PREPARATION COMPLETED FOR ALL NOISE RATIOS")
print("="*80)


# ============================================================================
# CELL 5: Tổng Kết và Báo Cáo Kết Quả
# ============================================================================

print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

# Đếm số lượng successful processing
successful = sum(1 for result in processing_results.values() if result['success'])
total = len(NOISE_RATIOS)

print(f"\nProcessing Summary:")
print(f"  Total noise ratios: {total}")
print(f"  Successfully processed: {successful}")
print(f"  Failed: {total - successful}")

# Chi tiết cho từng noise ratio
print(f"\nDetailed Results:")
for noise_ratio in NOISE_RATIOS:
    result = processing_results.get(noise_ratio, {})
    
    if result.get('success'):
        status = "✓ SUCCESS"
        verified = "✓" if result.get('verification_passed') else "⚠"
        print(f"\n  {status} - Noise Ratio {noise_ratio}:")
        print(f"    {verified} Verification: {'PASSED' if result.get('verification_passed') else 'FAILED'}")
        print(f"    📁 CSV files: {Path(result['paths']['train_csv']).parent}")
        print(f"    📊 EDA results: {result['eda_dir']}")
    else:
        status = "❌ FAILED"
        print(f"\n  {status} - Noise Ratio {noise_ratio}:")
        print(f"    Error: {result.get('error', 'Unknown error')}")

# Tạo summary table
print(f"\n{'='*80}")
print("VERIFICATION SUMMARY TABLE")
print(f"{'='*80}")

summary_data = []
for noise_ratio in NOISE_RATIOS:
    if noise_ratio in verification_results:
        train_stats = verification_results[noise_ratio]['train.csv']['stats']
        val_stats = verification_results[noise_ratio]['val.csv']['stats']
        test_stats = verification_results[noise_ratio]['test.csv']['stats']
        
        summary_data.append({
            'Noise Ratio': noise_ratio,
            'Train Samples': train_stats.get('total_samples', 0),
            'Train Noisy': train_stats.get('noisy_samples', 0),
            'Train Noise %': f"{train_stats.get('noise_ratio', 0)*100:.2f}%",
            'Val Samples': val_stats.get('total_samples', 0),
            'Val Noisy': val_stats.get('noisy_samples', 0),
            'Test Samples': test_stats.get('total_samples', 0),
            'Verified': '✓' if processing_results[noise_ratio].get('verification_passed') else '⚠'
        })

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print("\n")
    print(summary_df.to_string(index=False))
    
    # Save summary to CSV
    summary_csv_path = Path(config.data_dir) / "preprocessing_summary.csv"
    summary_df.to_csv(summary_csv_path, index=False)
    print(f"\n✓ Summary saved to: {summary_csv_path}")

# Recommendations
print(f"\n{'='*80}")
print("NEXT STEPS")
print(f"{'='*80}")

if successful == total:
    print("\n🎉 All datasets prepared successfully!")
    print("\nYou can now proceed with training experiments using these datasets.")
    print("\nRecommended next steps:")
    print("  1. Review the EDA visualizations to understand noise patterns")
    print("  2. Configure your training experiments in the main training script")
    print("  3. Run training with different alpha values for each noise ratio")
    
    print(f"\nData locations:")
    print(f"  📁 Images (shared): {config.data_dir}/images/")
    print(f"  📁 CSV files: {config.data_dir}/csvs/noise_<ratio>/")
    print(f"  📊 EDA results: {config.data_dir}/eda_analysis/noise_<ratio>/")
else:
    print("\n⚠ Some datasets failed to prepare properly.")
    print("\nPlease:")
    print("  1. Review the error messages above")
    print("  2. Fix any issues")
    print("  3. Re-run this notebook for the failed noise ratios")

print(f"\nCompleted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)


# ============================================================================
# CELL 6: Optional - Visualize Cross-Noise-Ratio Comparison
# ============================================================================

print("\n" + "="*80)
print("BONUS: CROSS-NOISE-RATIO VISUALIZATION")
print("="*80)

if successful == total and summary_data:
    # Tạo comparison plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Noise ratio comparison
    noise_ratios_actual_train = [d['Train Noise %'].rstrip('%') for d in summary_data]
    noise_ratios_actual_train = [float(x) for x in noise_ratios_actual_train]
    
    axes[0].bar(range(len(NOISE_RATIOS)), noise_ratios_actual_train, 
               color='coral', alpha=0.7, edgecolor='black')
    axes[0].plot(range(len(NOISE_RATIOS)), [r*100 for r in NOISE_RATIOS], 
                'bo-', linewidth=2, markersize=8, label='Target noise ratio')
    axes[0].set_xlabel('Noise Ratio Configuration', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Actual Noise Percentage (%)', fontsize=12, fontweight='bold')
    axes[0].set_title('Actual vs Target Noise Ratios', fontsize=14, fontweight='bold')
    axes[0].set_xticks(range(len(NOISE_RATIOS)))
    axes[0].set_xticklabels([f'{r:.1f}' for r in NOISE_RATIOS])
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Plot 2: Sample distribution
    train_samples = [d['Train Samples'] for d in summary_data]
    val_samples = [d['Val Samples'] for d in summary_data]
    test_samples = [d['Test Samples'] for d in summary_data]
    
    x = np.arange(len(NOISE_RATIOS))
    width = 0.25
    
    axes[1].bar(x - width, train_samples, width, label='Train', color='skyblue', edgecolor='black')
    axes[1].bar(x, val_samples, width, label='Val', color='lightgreen', edgecolor='black')
    axes[1].bar(x + width, test_samples, width, label='Test', color='salmon', edgecolor='black')
    
    axes[1].set_xlabel('Noise Ratio', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
    axes[1].set_title('Sample Distribution Across Splits', fontsize=14, fontweight='bold')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f'{r:.1f}' for r in NOISE_RATIOS])
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure
    comparison_plot_path = Path(config.data_dir) / "eda_analysis" / "noise_ratio_comparison.png"
    comparison_plot_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(comparison_plot_path, dpi=300, bbox_inches='tight')
    print(f"\n✓ Comparison plot saved to: {comparison_plot_path}")
    
    plt.show()
    
    print("\nThis plot shows:")
    print("  - Left: How close the actual noise ratio is to your target")
    print("  - Right: Sample distribution is consistent across noise ratios")
else:
    print("\n⚠ Skipping visualization due to incomplete processing")

print("\n" + "="*80)
print("NOTEBOOK EXECUTION COMPLETED")
print("="*80)

Thư mục parent: /mnt/c/Users/truon/learning/ptit/research/trung/M_10_01_2025/code_v2/project
Sys.path cập nhật: ['/mnt/c/Users/truon/learning/ptit/research/trung/M_10_01_2025/code_v2/project', '/mnt/c/Users/truon/learning/ptit/research/trung/M_10_01_2025/code_v2/project', '/mnt/c/Users/truon/learning/ptit/research/trung/M_10_01_2025/code_v2/project', '/mnt/c/Users/truon/learning/ptit/research/trung/M_10_01_2025/code_v2/project', '/home/trungsato/miniconda3/envs/self_ensembling/lib/python310.zip', '/home/trungsato/miniconda3/envs/self_ensembling/lib/python3.10', '/home/trungsato/miniconda3/envs/self_ensembling/lib/python3.10/lib-dynload', '', '/home/trungsato/.local/lib/python3.10/site-packages', '/home/trungsato/miniconda3/envs/self_ensembling/lib/python3.10/site-packages', '/tmp/tmpd3ej150f']
✓ Successfully imported project modules
Working directory: /mnt/c/Users/truon/learning/ptit/research/trung/M_10_01_2025/code_v2/project/notebooks
Parent directory: /mnt/c/Users/truon/learning/pti

Processing noise ratios:   0%|          | 0/4 [00:00<?, ?it/s]


PROCESSING NOISE RATIO: 0.2

Step 1: Creating dataset and CSV files (noise=0.2)
csv_dir: data_imagenet100_cmc/csvs/noise_0.2
train_csv_exists: True
⚠ CSV files for noise_ratio=0.2 already exist
  Location: data_imagenet100_cmc/csvs/noise_0.2
  Skipping data preparation...

Step 2: Verifying dataset (noise=0.2)

Verifying train.csv...
  ❌ train.csv verification FAILED
  Error: None

Verifying val.csv...
  ❌ val.csv verification FAILED
  Error: None

Verifying test.csv...
  ❌ test.csv verification FAILED
  Error: None

⚠ Some verification checks failed for noise_ratio=0.2
  Please review the output above for details.

Step 3: Running EDA analysis (noise=0.2)


2026-02-27 23:15:12,488 - src.eda_analysis - INFO - Starting complete EDA for noise_ratio=0.2


Running EDA analysis...
This will create:
  - Class distribution plots
  - Confusion matrices (counts and ratios)
  - Noise analysis report


...

2026-02-27 23:15:26,509 - src.eda_analysis - INFO - Starting complete EDA for noise_ratio=0.8


  ❌ test.csv verification FAILED
  Error: None

⚠ Some verification checks failed for noise_ratio=0.8
  Please review the output above for details.

Step 3: Running EDA analysis (noise=0.8)
Running EDA analysis...
This will create:
  - Class distribution plots
  - Confusion matrices (counts and ratios)
  - Noise analysis report


2026-02-27 23:15:29,215 - src.eda_analysis - INFO - Saved class distribution plot to data_imagenet100_cmc/eda_analysis/noise_0.8/class_distribution_noise_0.8.png



❌ Error processing noise_ratio=0.8: index 89 is out of bounds for axis 1 with size 10

DATA PREPARATION COMPLETED FOR ALL NOISE RATIOS

FINAL SUMMARY

Processing Summary:
  Total noise ratios: 4
  Successfully processed: 0
  Failed: 4

Detailed Results:

  ❌ FAILED - Noise Ratio 0.2:
    Error: index 82 is out of bounds for axis 1 with size 10

  ❌ FAILED - Noise Ratio 0.4:
    Error: index 80 is out of bounds for axis 1 with size 10

  ❌ FAILED - Noise Ratio 0.6:
    Error: index 89 is out of bounds for axis 1 with size 10

  ❌ FAILED - Noise Ratio 0.8:
    Error: index 89 is out of bounds for axis 1 with size 10

VERIFICATION SUMMARY TABLE


 Noise Ratio  Train Samples  Train Noisy Train Noise %  Val Samples  Val Noisy  Test Samples Verified
         0.2         114020        22804        20.00%        12669       2534          5000        ⚠
         0.4         114020        45608        40.00%        12669       5068          5000        ⚠
         0.6         114020        68412 

Traceback (most recent call last):
  File "/tmp/ipykernel_1612/1435049311.py", line 374, in <module>
    perform_complete_eda(paths, noise_ratio, str(eda_dir))
  File "/mnt/c/Users/truon/learning/ptit/research/trung/M_10_01_2025/code_v2/project/src/eda_analysis.py", line 281, in perform_complete_eda
    plot_noise_confusion_matrix_counts(csv_paths[csv_key], noise_ratio,
  File "/mnt/c/Users/truon/learning/ptit/research/trung/M_10_01_2025/code_v2/project/src/eda_analysis.py", line 105, in plot_noise_confusion_matrix_counts
    confusion_matrix[orig, noisy] += 1
IndexError: index 89 is out of bounds for axis 1 with size 10
